<a href="https://colab.research.google.com/github/saminsiddiqui08-beep/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saminsiddiqui08-beep/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

-> I am selecting Lane D: Under-Clicked Visible Pages. This lane models the relationship between ranking position and expected CTR to isolated pages that underperform their ranking potential.

By predicting the Expected CTR conditioned on specific aspects, we calculate the CTR deficit (Expected CTR - Actual CTR) and rank pages by Unrealized Traffic Oppurtunity. This allows us to make targetted improvements to the isolated pages for maximum impact.

In [7]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Setup: Clone repository to Colab's local disk if running in Colab
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/saminsiddiqui08-beep/flyrank-ml-internship"
    REPO_DIR = "flyrank-ml-internship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Filtering to visible pages (impressions >= 100)
visible_df = df[df["impressions_90d"] >= 100].copy()

print(f"Total dataset rows: {len(df):,}")
print(f"Visible candidate pages (>=100 impressions): {len(visible_df):,} ({len(visible_df)/len(df)*100:.1f}%)")

Total dataset rows: 30,000
Visible candidate pages (>=100 impressions): 22,006 (73.4%)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

->
1. It improves the content optimization prioritization decision. This work prioritizes pages with the highest uncaptured traffic potential.


2. SEO specialists, content strategists, and growth marketing teams responsible for rewriting title tags, meta descriptions, and SERP search snippets.


3. High-ranking pages with severe snippet underperformance remain unflagged which leaks thousands of potential organic visito

In [8]:
# Measuring total search exposure in top ranking positions
high_vis = visible_df[visible_df["avg_position"] <= 20]
total_impressions_at_stake = high_vis["impressions_90d"].sum()

print(f"High-visibility candidate pages (Top 20 positions): {len(high_vis):,}")
print(f"Total 90-day search impressions at stake: {total_impressions_at_stake:,.0f}")


High-visibility candidate pages (Top 20 positions): 15,091
Total 90-day search impressions at stake: 119,608,898


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

->
  1. Pages ranking on Page 1 average a ~35.5% CTR, which drops sharply to ~25.6% in striking positions (ranks 4–10) and falls to ~5.5% for deeper ranks.

  2. Among pages ranking in the top 10 positions, the 25th percentile CTR is 0.09 while the 75th percentile is 0.46 (IQR: 0.37). This wide spread confirms that rank alone does not dictate CTR; page-level snippet and intent attributes drive substantial performance variance.

  3. Over 11.7 million search impressions (11,745,262) are concentrated in top-10 ranking pages that fall into the lowest quartile of CTR, representing a concrete, high-impact target pool for search snippet optimization.

In [9]:
# 1. Mean CTR across ranking tiers
tier_ctr = visible_df.groupby("position_tier")["ctr"].mean()
print("--- 1. Mean CTR by Position Tier ---")
print(tier_ctr.round(4))

# 2. Top 10 CTR spread (variance)
top_tier = visible_df[visible_df["avg_position"] <= 10]
p25 = top_tier["ctr"].quantile(0.25)
p75 = top_tier["ctr"].quantile(0.75)
print(f"\n--- 2. Top-10 CTR Spread ---\n25th Percentile: {p25:.4f} | 75th Percentile: {p75:.4f} (IQR: {p75-p25:.4f})")

# 3. Impression volume in underperforming top 10 pages
underclicked_pool = top_tier[top_tier["ctr"] < p25]["impressions_90d"].sum()
print(f"\n--- 3. Total Impressions in Bottom-Quartile Top-10 Pages ---\n{underclicked_pool:,.0f} impressions")


--- 1. Mean CTR by Position Tier ---
position_tier
deep        0.0554
page_1      0.3548
page_3_5    0.1424
striking    0.2558
top_3       0.3341
Name: ctr, dtype: float64

--- 2. Top-10 CTR Spread ---
25th Percentile: 0.0900 | 75th Percentile: 0.4600 (IQR: 0.3700)

--- 3. Total Impressions in Bottom-Quartile Top-10 Pages ---
11,745,262 impressions


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

->
* What this work can claim (Observed, Directional, Decision-Support): We measure observed  relationships between ranking position, content characteristics, and Click-Through Rate. We provide directional decision-support by ranking URLs where the gap between actual CTR and cohort-expected CTR is statistically largest. We identify high-exposure pages that underperform peer benchmarks, allowing content teams to prioritize editorial review efficiently.

* What this work CAN NEVER claim (Causal Proof & Algorithm Prediction): We do not claim causal proof (e.g., we cannot claim that rewriting a title tag guarantees an $X\%$ increase in CTR). We do not claim to predict or reverse-engineer Google's ranking algorithm. All outputs are statistical heuristics intended solely for human workflow prioritization, not automated search engine manipulation.

In [10]:
preliminary_features = ["avg_position", "impressions_90d", "word_count", "content_age_days"]

# Confirming that outcome/label proxies are excluded
assert "trend_direction" not in preliminary_features, "Leakage detected: target label present!"
assert "trend_pct" not in preliminary_features, "Leakage detected: label proxy present!"

print("Integrity verification passed:")
print(f"- Preliminary features: {preliminary_features}")
print("- No target leakage columns detected.")
print("- All metrics reflect observational, decision-support boundaries.")

Integrity verification passed:
- Preliminary features: ['avg_position', 'impressions_90d', 'word_count', 'content_age_days']
- No target leakage columns detected.
- All metrics reflect observational, decision-support boundaries.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.